# pandas: Series and DataFrames, from basic to advanced

The [previous block](03-lecture-notebooks-python-recap-and-numpy.ipynb) computed with NumPy arrays: fast, but one type per array and positions instead of names. Real data are **tables**: columns with names and different types (district as text, price as a number, last review as a date), rows that describe one thing each. **pandas** builds such tables on top of NumPy. This block goes from the basic objects to the operations that answer most everyday questions: selecting, transforming, grouping, reshaping and combining tables, written as one readable chain. It ends with how to use AI coding assistants for this work without trusting them blindly.

All examples use the Berlin Airbnb listings (Inside Airbnb, snapshot of 26 June 2026) and the monthly review counts of the same snapshot.

```mermaid
flowchart LR
  S["Series and DataFrame<br/>index, columns, dtype"] --> Q["Select<br/>loc, iloc, masks,<br/>missing values"]
  Q --> T["Transform<br/>arithmetic, .str, .dt,<br/>map, apply"]
  T --> G["Summarise<br/>sort, value_counts,<br/>groupby, pivot_table"]
  G --> M["Combine<br/>merge, concat"]
  M --> K["Method chaining"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Series and DataFrame: index, columns and `dtype`; reading a file

pandas has two main types:

- A **Series** is one column: a NumPy-like array of values with one `dtype`, plus an **index** of labels, one per value.
- A **DataFrame** is a table: several Series that share the same index, one per **column**. Each column has its own `dtype`.

The index is what NumPy does not have: rows keep their labels when you filter or sort, and two Series are aligned by label, not by position, when you combine them. A list of dictionaries, one per record, becomes a DataFrame directly.

In [2]:
import numpy as np
import pandas as pd

records = [
    {"district": "Pankow", "nightly_price": 75, "available": True},
    {"district": "Mitte", "nightly_price": 90, "available": False},
    {"district": "Neukölln", "nightly_price": None, "available": True},
]
small = pd.DataFrame(records)
print(small.dtypes)
small

district             str
nightly_price    float64
available           bool
dtype: object


,district,nightly_price,available
0,Pankow,75.0,True
1,Mitte,90.0,False
2,Neukölln,NaN,True


In [3]:
price = small["nightly_price"]          # one column: a Series
print(type(price).__name__, price.dtype)
print(price.index.tolist(), price.to_numpy())

Series float64
[0, 1, 2] [75. 90. nan]


The missing price became `NaN` and the column `float64`, as in NumPy. Text columns have the dtype `str` in pandas 3 (older versions showed `object`).

**Reading a file.** `pd.read_csv`, `pd.read_excel`, `pd.read_json` and `pd.read_parquet` turn a file into a DataFrame. Parquet stores the column types with the data, so dates arrive as dates and numbers as numbers. A CSV file is plain text, and pandas guesses the types: give `sep=";"` and `decimal=","` for German Excel exports, `dtype={"postal_code": str}` for identifiers with leading zeros, and `parse_dates=[...]` for dates.

**Before calculating, define what one row is.** Here one row is one listing in the snapshot, not a booking, a host or a flat.

In [4]:
listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
print(listings.shape)                         # (rows, columns)
print(listings.dtypes.value_counts())         # how many columns of each type
listings[["id", "district", "room_type", "accommodates", "price", "minimum_nights", "last_review"]].head()

(12776, 42)
float64           19
int64             10
str                8
datetime64[ns]     3
object             2
Name: count, dtype: int64


,id,district,room_type,accommodates,price,minimum_nights,last_review
0,3176,Pankow,Entire home/apt,2,160.71,2.0,2025-08-09
1,9991,Pankow,Entire home/apt,7,193.33,6.0,2020-01-04
2,14325,Pankow,Entire home/apt,1,NaN,180.0,2023-11-30
3,17904,Friedrichshain-Kreuzberg,Entire home/apt,2,10.88,93.0,2026-05-31
4,20858,Pankow,Entire home/apt,4,372.67,3.0,2026-05-29


`head()` shows the first rows, `listings.info()` the columns with their types and the number of values present, `listings.describe()` summary statistics of the numeric columns. The index is `0, 1, 2, ...` (a `RangeIndex`); `set_index("id")` would make the listing id the row label.

> **In practice.** Check `shape` and `dtypes` right after reading a file. A numeric column read as `str` or a date read as text is the most common reason why a later step fails or, worse, gives a wrong answer silently.

## Selecting rows and columns with `loc`, `iloc` and Boolean masks; missing values

- `df["col"]` selects one column as a Series; `df[["a", "b"]]` (a list of names) a smaller DataFrame.
- `.loc[rows, columns]` selects by **labels**; `.iloc[rows, columns]` by **positions**, as in NumPy.
- A comparison on a column gives a Boolean Series, a **mask**. `df[mask]` or `df.loc[mask, columns]` keeps the rows where it is `True`. Combine masks with `&`, `|` and `~`, with parentheses around each comparison. `isin([...])` and `between(a, b)` save long chains of `|` and `&`.

In [5]:
print(listings.loc[0, "district"], "|", listings.iloc[0, 9])      # label 0 / position 9 (the district column)

by_id = listings.set_index("id")
print(by_id.loc[3176, ["district", "room_type", "price"]].to_dict())

inner = listings["district"].isin(["Mitte", "Friedrichshain-Kreuzberg", "Pankow"])
for_four = listings["accommodates"].between(4, 6)                 # both ends included
print(inner.sum(), (inner & for_four).sum())

Pankow | Pankow
{'district': 'Pankow', 'room_type': 'Entire home/apt', 'price': 160.71}
7428 2254


**Labels survive filtering and sorting; positions do not.** After `sort_values`, `.iloc[0]` is the first row on the page, while `.loc[0]` is still the row whose label is 0, wherever it now stands. Mixing them up selects the wrong row without any error.

**Missing values.** pandas marks a missing value as `NaN` (or `NaT` for dates, `None` or `NaN` in text columns). Find them with `.isna()` and `.notna()`, never with `== None` or `== np.nan` (both are always `False` for `NaN`). `count()` counts the values that are present, `len()` counts rows. Summary methods such as `mean()` and `median()` skip missing values; arithmetic with them gives `NaN`.

In [6]:
print(len(listings), listings["price"].count())
listings.isna().mean().sort_values(ascending=False).head(5).round(3)   # share missing, worst columns first

12776 8441


bathrooms                  0.521
beds                       0.490
price                      0.339
estimated_revenue_l365d    0.339
license                    0.311
dtype: float64

About a third of the listings have no price, and many can only be booked for months. As in the previous block, the course compares prices only for **short stays**: a price present and a minimum stay below 28 nights.

In [7]:
short = listings[listings["price"].notna() & (listings["minimum_nights"] < 28)]
print(len(short), "short-stay listings with a price")

6701 short-stay listings with a price


> **In practice.** Decide for each analysis what a missing value means. `dropna()` removes rows, `fillna(0)` claims a value: a missing price is not a price of €0, but a listing without a review row in a review table really has zero reviews. Write the decision next to the code.

> 📖 **Read more:** [extras/12-pandas-indexing-and-selection.ipynb](extras/12-pandas-indexing-and-selection.ipynb) (Python Data Science Handbook): why `.loc` and `.iloc` exist, slicing, masking and fancy indexing.

## Creating and transforming columns: arithmetic, `.str`, `.dt`, `map` and `apply`

New columns come from existing ones, mostly with vectorised operations as in NumPy. `df.assign(name=...)` returns a new table with the column added; `df["name"] = ...` adds it in place.

- **Arithmetic** works element by element and aligned by the index.
- **`.str`** applies string methods to every value of a text column: `lower`, `contains`, `startswith`, `replace`, `split`, `len`, regular expressions with `match` and `extract`.
- **`.dt`** gives the parts of a date column: `year`, `month`, `dayofweek`, `day_name()`; subtracting two dates gives a duration, whose `.dt.days` is a number.
- **`map`** replaces each value by looking it up in a dictionary (or by applying a function to it).
- **`apply`** runs a Python function on every row or every value. It is flexible but slow, because it is a Python loop: use it only when no vectorised method exists.

In [8]:
snapshot = pd.Timestamp("2026-06-26")
short_labels = {"Entire home/apt": "entire", "Private room": "private",
                "Shared room": "shared", "Hotel room": "hotel"}

short = short.assign(
    price_per_guest=short["price"] / short["accommodates"],                       # arithmetic
    balcony=short["name"].str.contains("balcony|balkon", case=False),             # .str, with a regular expression
    review_year=short["last_review"].dt.year,                                     # .dt
    days_since_review=(snapshot - short["last_review"]).dt.days,                  # a duration in days
    room=short["room_type"].map(short_labels),                                    # map with a dictionary
)
short[["name", "room", "accommodates", "price", "price_per_guest", "balcony", "review_year", "days_since_review"]].head()

,name,room,accommodates,price,price_per_guest,balcony,review_year,days_since_review
0,Fabulous Flat in great Location,entire,2,160.71,80.355000,False,2025.0,321.0
1,Geourgeous flat - outstanding views,entire,7,193.33,27.618571,False,2020.0,2365.0
4,Designer Loft in Berlin Mitte,entire,4,372.67,93.167500,False,2026.0,28.0
6,"""Prenzel Garten"" mit Terrasse",entire,5,243.50,48.700000,False,2026.0,4.0
8,PEACEFUL FLAT WITH VINTAGE TOUCH IN PRENZLAUER...,entire,8,212.50,26.562500,False,2026.0,1.0


In [9]:
print(short["balcony"].mean().round(3), "of the short-stay titles mention a balcony")
print(short["price_per_guest"].median(), "median price per guest and night")
print(short["review_year"].value_counts().sort_index().tail(3))

0.051 of the short-stay titles mention a balcony
52.25 median price per guest and night
review_year
2024.0     191
2025.0     615
2026.0    5319
Name: count, dtype: int64


`apply` with `axis=1` passes one row at a time to a function. The rule below gives a rough "size class"; it could be written without `apply` (with `np.select` or `pd.cut`), and on large tables it should be.

In [10]:
def size_class(row):
    """Classify a listing by room type and number of guests."""
    if row["room_type"] != "Entire home/apt":
        return "room"
    return "large flat" if row["accommodates"] >= 4 else "small flat"


short["size_class"] = short.apply(size_class, axis=1)
short["size_class"].value_counts()

size_class
large flat    2645
small flat    2043
room          2013
Name: count, dtype: int64

> **In practice.** pandas 3 uses *Copy-on-Write*: a filtered table such as `short` behaves as an independent copy, and changing it never changes `listings`. To change some rows of a column, select rows and column in one step: `df.loc[mask, "col"] = value`. A chained form such as `df[mask]["col"] = value` changes a temporary copy and has no effect.

## Sorting, `value_counts`, `groupby` with `agg`, and pivot tables

**Sorting and counting.** `sort_values` sorts by one or more columns (`ascending=False` from the largest); `nlargest(n, "col")` is a shortcut. `value_counts()` counts the categories of a column, `normalize=True` gives shares.

In [11]:
print(short.nlargest(3, "price")[["district", "room_type", "accommodates", "price"]])
listings["room_type"].value_counts(normalize=True).round(3)

                    district        room_type  accommodates    price
8434   Steglitz - Zehlendorf  Entire home/apt             7  10025.0
12710                Spandau  Entire home/apt             5   7999.2
9515      Treptow - Köpenick  Entire home/apt            16   4458.0


room_type
Entire home/apt    0.692
Private room       0.294
Hotel room         0.007
Shared room        0.007
Name: proportion, dtype: float64

**`groupby`.** Most questions are "per something": per district, per room type, per host. `groupby` **splits** the rows into groups by the values of one or more columns, **applies** a summary to each group and **combines** the results into a new table, the *split-apply-combine* pattern.

```mermaid
flowchart LR
  T["short stays<br/>6,701 rows"] -->|"split by district"| G1["Mitte rows"]
  T --> G2["Pankow rows"]
  T --> G3["... 12 groups"]
  G1 -->|"apply: median price"| R1["187.0"]
  G2 -->|"apply: median price"| R2["174.0"]
  G3 -->|"apply"| R3["..."]
  R1 --> C["combine: one row per district"]
  R2 --> C
  R3 --> C
```

`agg` computes several summaries at once. With **named aggregation**, `new_name=("column", "function")`, the result has readable column names.

In [12]:
by_district = (
    short.groupby("district")
    .agg(listings=("id", "size"),
         median_price=("price", "median"),
         median_per_guest=("price_per_guest", "median"),
         share_balcony=("balcony", "mean"))
    .sort_values("median_price", ascending=False)
    .round(2)
)
by_district

,listings,median_price,median_per_guest,share_balcony
district,,,,
Mitte,1537,187.00,58.14,0.05
Pankow,1033,174.00,53.25,0.07
Friedrichshain-Kreuzberg,1337,160.00,53.38,0.04
Charlottenburg-Wilm.,760,149.30,53.88,0.05
Tempelhof - Schöneberg,478,137.25,47.85,0.04
Lichtenberg,226,132.00,47.00,0.07
Steglitz - Zehlendorf,174,130.62,46.73,0.03
Neukölln,556,130.00,50.50,0.08
Spandau,105,126.50,49.08,0.01


Mitte is the most expensive district for a night (median €187), Reinickendorf the cheapest (€99.54). The **median** is reported because prices are skewed: 25 short-stay listings ask for more than €1,000 a night and pull the mean (€194) far above the typical price.

![Price distributions of short-stay listings by room type, with mean and median marked](../figures/price_by_room_type.png)

**Pivot tables.** A `groupby` over two columns gives a long table with one row per combination. `pivot_table` spreads one of the grouping columns across the columns, which is easier to read: rows by district, columns by room type, cells the median price. `margins=True` adds the totals over all rows and columns.

In [13]:
two_types = short[short["room_type"].isin(["Entire home/apt", "Private room"])]
prices_table = two_types.pivot_table(index="district", columns="room_type", values="price",
                                     aggfunc="median", margins=True, margins_name="All")
prices_table.round(0)

room_type,Entire home/apt,Private room,All
district,,,
Charlottenburg-Wilm.,180.0,103.0,149.0
Friedrichshain-Kreuzberg,187.0,100.0,162.0
Lichtenberg,164.0,81.0,133.0
Marzahn - Hellersdorf,142.0,74.0,126.0
Mitte,212.0,117.0,192.0
Neukölln,160.0,93.0,132.0
Pankow,198.0,98.0,174.0
Reinickendorf,121.0,74.0,100.0
Spandau,168.0,80.0,124.0


In every district an entire home costs between 1.6 and 2.1 times as much as a private room. The `All` row is the median over all districts (€185 and €97), not the mean of the twelve district medians. With all four room types, cells for combinations without a listing (no short-stay shared room in Spandau, for example) would be `NaN`: `aggfunc="size"` shows how many rows stand behind each cell, which you should check before trusting a median of a handful of listings.

![Listing counts and median short-stay prices by district](../figures/listings_by_district.png)

> 📖 **Read more:** [extras/14-pandas-groupby.ipynb](extras/14-pandas-groupby.ipynb) (Python Data Science Handbook): split-apply-combine in depth: `aggregate`, `filter`, `transform` and `apply`.

## Combining tables with `merge` and `concat`

Data come in several tables. **`merge`** joins two tables on a **key**, a column that identifies the same thing in both, like a database join (Session 3 does the same in SQL). The `how` argument decides what happens to rows without a partner:

```mermaid
flowchart LR
  L["left table<br/>listings"] --> J{"merge on the key<br/>id = listing_id"}
  R["right table<br/>reviews in 2025"] --> J
  J -->|"how='inner'"| I["only keys in both"]
  J -->|"how='left'"| LL["all listings;<br/>NaN where no reviews"]
  J -->|"how='right'"| RR["all review rows"]
  J -->|"how='outer'"| O["all keys of both"]
```

The review table has one row per listing and month with the number of reviews written. How many reviews did each listing get in 2025? First aggregate to one row per listing, then merge.

In [14]:
reviews = pd.read_parquet("case-study/data/airbnb/reviews_monthly.parquet")
print(reviews.shape)
print(reviews.head(3))

reviews_2025 = (
    reviews[reviews["month"].dt.year == 2025]
    .groupby("listing_id", as_index=False)["n_reviews"].sum()
    .rename(columns={"n_reviews": "reviews_2025"})
)
print(len(reviews_2025), "listings with at least one review in 2025")

(226557, 3)
   listing_id      month  n_reviews
0        3176 2009-06-01          1
1        3176 2010-11-01          2
2        3176 2010-12-01          1
6409 listings with at least one review in 2025


In [15]:
merged = listings.merge(reviews_2025, left_on="id", right_on="listing_id", how="left",
                        validate="one_to_one", indicator=True)
print(len(listings), len(merged))
print(merged["_merge"].value_counts())

12776 12776
_merge
left_only     6395
both          6381
right_only       0
Name: count, dtype: int64


The left merge keeps all 12,776 listings; 6,381 found a partner, and the others get `NaN` in `reviews_2025`. Here a missing row really means "no review in 2025", so `fillna(0)` is correct. `validate="one_to_one"` makes pandas stop with an error if a key appears twice, which would silently multiply rows; `indicator=True` adds the `_merge` column that shows where each row came from.

The review table has 6,409 listings with a review in 2025, 28 more than matched: those listings are not in the listings table. An inner merge would have dropped them without notice, and also the 6,395 listings without a review in 2025.

In [16]:
merged["reviews_2025"] = merged["reviews_2025"].fillna(0)
merged.groupby("district")["reviews_2025"].sum().sort_values(ascending=False).head(5)

district
Mitte                       32541.0
Friedrichshain-Kreuzberg    31179.0
Pankow                      21587.0
Charlottenburg-Wilm.        12827.0
Neukölln                     8480.0
Name: reviews_2025, dtype: float64

**`concat`** stacks tables with the same columns on top of each other (or side by side with `axis=1`), for example the data of several months or two subsets you want to compare in one table. `keys=` labels where each part came from.

In [17]:
cheapest = short.nsmallest(2, "price")[["district", "room_type", "price"]]
dearest = short.nlargest(2, "price")[["district", "room_type", "price"]]
pd.concat([cheapest, dearest], keys=["cheapest", "most expensive"])

district        room_type     price
cheapest       107                      Pankow     Private room      9.03
               2574   Friedrichshain-Kreuzberg     Private room     10.47
most expensive 8434      Steglitz - Zehlendorf  Entire home/apt  10025.00
               12710                   Spandau  Entire home/apt   7999.20

> **In practice.** Before a merge, check that the key is unique where you expect it (`df["id"].is_unique`); after it, compare the row count with the left table. A many-to-many merge on a key with duplicates is the most common way to inflate a table without an error.

> 📖 **Read more:** [extras/13-pandas-merge-and-join.ipynb](extras/13-pandas-merge-and-join.ipynb) (Python Data Science Handbook): one-to-one, many-to-one and many-to-many joins, keys with different names, and a worked example with US state data.

## Method chaining

The steps above can be written as one **chain**: each method returns a new table, so the next method can be called on it directly. Wrapped in parentheses, a chain has one step per line and reads from top to bottom like a recipe. There are no intermediate names to keep track of and no half-finished tables left in the kernel.

`loc` and `assign` accept a function (`lambda d: ...`) that receives the table *at that point of the chain*, so a filter can use a column created in an earlier step. The chain below answers: in which districts is a short-stay entire home for four or more guests cheapest per guest?

In [18]:
per_guest_ranking = (
    listings
    .loc[lambda d: d["price"].notna() & (d["minimum_nights"] < 28)]         # short stays with a price
    .loc[lambda d: (d["room_type"] == "Entire home/apt") & (d["accommodates"] >= 4)]
    .assign(price_per_guest=lambda d: d["price"] / d["accommodates"])
    .groupby("district")
    .agg(flats=("id", "size"), median_per_guest=("price_per_guest", "median"))
    .query("flats >= 50")                                                     # enough flats to compare
    .sort_values("median_per_guest")
)
per_guest_ranking.round(2)

,flats,median_per_guest
district,,
Steglitz - Zehlendorf,69,35.75
Treptow - Köpenick,146,36.72
Lichtenberg,84,38.62
Neukölln,167,38.92
Tempelhof - Schöneberg,169,41.83
Friedrichshain-Kreuzberg,540,42.94
Pankow,491,45.55
Charlottenburg-Wilm.,229,47.25
Mitte,664,48.14


> **In practice.** Build a chain one line at a time and run it after each new line. If a result looks wrong, comment out the last steps to see the intermediate table, or insert `.pipe(lambda d: print(d.shape) or d)` to print the row count. Very long chains are as hard to read as long cells: split them where an intermediate table has a meaning of its own.

> 📖 **Read more:** [extras/11-pandas-introduction.ipynb](extras/11-pandas-introduction.ipynb) (Microsoft Data Science for Beginners): a beginner walk-through of Series and DataFrames; some examples need updating for current pandas.

## Tips for using AI coding assistants

AI assistants are allowed in this course, and many data teams use them every day. They write plausible pandas code quickly, and plausible is not the same as correct: a filter can keep the wrong rows, a merge can duplicate rows, `fillna(0)` can turn a missing price into a free flat, and none of these raises an error.

- **Ask precisely.** Name the input (columns and their types), the output you want and the edge cases: "short stays are listings with a price and `minimum_nights < 28`; keep listings without reviews".
- **Check on a small table.** Run the code on three or four rows whose answer you can compute by hand, like `small` above, before you run it on 12,776 rows.
- **Check the real result.** Compare row counts before and after each filter and merge, and compute one number in a second, independent way (a mask and `len()` instead of `value_counts()`).
- **Understand every line.** If you cannot explain a line, ask the assistant to explain it, or look it up with `help()`. You are responsible for the code you submit.
- **Document the use.** Record which tool you used and for what, as the course requires for the final project (HTW declaration).
- **Learn first.** In the exercises, solve the problem yourself before asking an assistant. Use it to explain an error or an idea, not to replace the practice that builds the skill.

## Check your understanding

1. What does a Series have that a NumPy array does not, and why does it matter when you sort or filter a table?
2. After `df = df.sort_values("price")`, what is the difference between `df.loc[0]` and `df.iloc[0]`?
3. Why is `fillna(0)` right for the number of reviews in 2025 after the merge, but wrong for a missing price?
4. Which arguments of `merge` protect you from duplicated keys and from losing rows without noticing?
5. Rewrite in words what the chain of the last section does, one line per step. Which step would you check first if a district was missing from the result?

## Further reading

- McKinney, W. (2022). *Python for Data Analysis*, 3rd edition, chapters 5 (getting started with pandas), 7 (data cleaning and preparation), 8 (join, combine and reshape) and 10 (aggregation and group operations). Open access. https://wesmckinney.com/book/
- pandas development team (2026). *Getting started tutorials* and *User guide*: indexing and selecting data, working with missing data, working with text data, group by, reshaping and pivot tables, merge, join and concatenate, Copy-on-Write. https://pandas.pydata.org/docs/user_guide/index.html
- VanderPlas, J. (2016). *Python Data Science Handbook*, chapter 3 (data manipulation with pandas). O'Reilly. https://jakevdp.github.io/PythonDataScienceHandbook/ (the source of extras 12–14).